# Dropout in code: regression and classification
The same 128-128 network with and without `Dropout` layers, on a small regression problem and a small
classification problem. Results go to `data/` for the figures in `images/`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

tf.config.experimental.enable_op_determinism()   # same numbers on every run
os.makedirs("data", exist_ok=True)

## 1. Regression
20 training points and 20 test points on the line y = x, plus noise.

In [ ]:
rng = np.random.default_rng(1)
N = 20
X_train = np.linspace(-1, 1, N).reshape(-1, 1)
y_train = X_train[:, 0] + rng.normal(0, 0.3, N)
X_test = np.linspace(-1, 1, N).reshape(-1, 1)
y_test = X_test[:, 0] + rng.normal(0, 0.3, N)
pd.DataFrame({"x": X_train[:, 0], "y_train": y_train, "y_test": y_test}).round(4).to_csv(
    "data/regression_points.csv", index=False)

In [ ]:
def build(p, n_inputs, output_activation, loss, seed=0):
    """Two hidden layers of 128 ReLU nodes; a Dropout layer with rate p after each (none if p = 0)."""
    keras.utils.set_random_seed(seed)
    layers = [keras.Input(shape=(n_inputs,))]
    for _ in range(2):
        layers.append(keras.layers.Dense(128, activation="relu"))
        if p > 0:
            layers.append(keras.layers.Dropout(p))     # switches off a fraction p of the nodes above
    layers.append(keras.layers.Dense(1, activation=output_activation))
    model = keras.Sequential(layers)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss=loss,
                  metrics=["accuracy"] if loss == "binary_crossentropy" else [])
    return model

build(0.2, 1, "linear", "mse").summary()

In [ ]:
x_line = np.linspace(-1, 1, 201).reshape(-1, 1)
reg_curves, reg_hist, reg_scores = {"x": x_line[:, 0]}, {}, []
for p in [0, 0.2, 0.5, 0.75]:
    model = build(p, 1, "linear", "mse")
    h = model.fit(X_train, y_train, epochs=500, validation_data=(X_test, y_test), verbose=0).history
    train_mse = model.evaluate(X_train, y_train, verbose=0)   # dropout is off when evaluating
    test_mse = model.evaluate(X_test, y_test, verbose=0)
    reg_scores.append({"p": p, "train_mse": train_mse, "test_mse": test_mse})
    reg_curves[f"p={p}"] = model.predict(x_line, verbose=0).ravel()
    reg_hist[f"loss p={p}"], reg_hist[f"val_loss p={p}"] = h["loss"], h["val_loss"]
reg_scores = pd.DataFrame(reg_scores)
print(reg_scores.round(4))
pd.DataFrame(reg_curves).round(4).to_csv("data/regression_curves.csv", index=False)
pd.DataFrame(reg_hist).round(5).to_csv("data/regression_history.csv", index_label="epoch")
reg_scores.round(5).to_csv("data/regression_scores.csv", index=False)

## 2. Classification
Two overlapping clouds of points, 200 for training and 200 for validation.

In [ ]:
rng = np.random.default_rng(3)
def two_clouds(n):
    y = rng.integers(0, 2, n)
    X = rng.normal(0, 1, (n, 2)) + np.where(y[:, None] == 1, 0.6, -0.6)   # class 1 around (0.6, 0.6), class 0 around (-0.6, -0.6)
    return X, y
Xc_train, yc_train = two_clouds(200)
Xc_test, yc_test = two_clouds(200)
pd.DataFrame({"x1": Xc_train[:, 0], "x2": Xc_train[:, 1], "y": yc_train}).round(4).to_csv(
    "data/classification_points.csv", index=False)

In [ ]:
xx, yy = np.meshgrid(np.linspace(-4, 4, 161), np.linspace(-4, 4, 161))
grid = np.c_[xx.ravel(), yy.ravel()]
cls_grid, cls_hist, cls_scores = {"x1": grid[:, 0], "x2": grid[:, 1]}, {}, []
for p in [0, 0.2, 0.5]:
    model = build(p, 2, "sigmoid", "binary_crossentropy")
    h = model.fit(Xc_train, yc_train, epochs=500, validation_data=(Xc_test, yc_test), verbose=0).history
    _, train_acc = model.evaluate(Xc_train, yc_train, verbose=0)
    _, val_acc = model.evaluate(Xc_test, yc_test, verbose=0)
    cls_scores.append({"p": p, "train_accuracy": train_acc, "val_accuracy": val_acc,
                       "final_val_loss": h["val_loss"][-1]})
    cls_grid[f"p={p}"] = model.predict(grid, verbose=0).ravel()
    cls_hist[f"loss p={p}"], cls_hist[f"val_loss p={p}"] = h["loss"], h["val_loss"]
cls_scores = pd.DataFrame(cls_scores)
print(cls_scores.round(3))
pd.DataFrame(cls_grid).round(4).to_csv("data/classification_grid.csv", index=False)
pd.DataFrame(cls_hist).round(5).to_csv("data/classification_history.csv", index_label="epoch")
cls_scores.round(5).to_csv("data/classification_scores.csv", index=False)